In [ ]:
import gradio as gr
import os
from transformers import BertForSequenceClassification, BertTokenizer, pipeline

# Load model path
model_path = "/content/news_topic_model"

# Load model & tokenizer
model = BertForSequenceClassification.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained(model_path)

# Pipeline (returns all scores)
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    top_k=None
)

def predict(text):
    if not text.strip():
        return "Please enter some text."

    results = classifier(text)[0]
    results = sorted(results, key=lambda x: x['score'], reverse=True)

    output = "Predictions:\n\n"
    for res in results:
        label_id = int(res['label'].split("_")[-1])
        label_name = model.config.id2label[label_id]
        score = round(res['score'] * 100, 2)

        output += f"{label_name}: {score}%\n"

    return output

# UI
with gr.Blocks() as demo:

    gr.Markdown("# News Topic Classifier (BERT)")
    gr.Markdown("Classify headlines into World, Sports, Business, or Sci/Tech")

    input_text = gr.Textbox(
        lines=3,
        placeholder="Type your news headline here..."
    )

    with gr.Row():
        predict_btn = gr.Button("  Analyze")
        clear_btn = gr.Button(" Clear ")

    output_text = gr.Textbox(label="Result", interactive=False)

    gr.Examples(
        examples=[
            ["Stock markets surge after tech rally"],
            ["New AI breakthrough in robotics"],
            ["Football team wins championship"],
            ["Global leaders meet for climate summit"]
        ],
        inputs=input_text
    )

    predict_btn.click(fn=predict, inputs=input_text, outputs=output_text)
    input_text.submit(fn=predict, inputs=input_text, outputs=output_text)
    clear_btn.click(lambda: ("", ""), outputs=[input_text, output_text])

# Launch
if __name__ == "__main__":
    demo.launch(share=True)